In [3]:
import urllib.request
import zipfile
import os
from pathlib import Path


url ="https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "../text/sms_spam_collection.zip"
extracted_path = "../text/sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print('Already there!')
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, 'wb') as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

In [4]:
download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

File downloaded and saved as ../text/sms_spam_collection/SMSSpamCollection.tsv


In [5]:
import pandas as pd

df = pd.read_csv(data_file_path, sep='\t', header=None, names=['Label', "Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
df['Label'].value_counts()

Label
ham     4825
spam     747
Name: count, dtype: int64

In [9]:
def create_balanced_dataset(df):
    num_spam = df[df['Label'] == 'spam'].shape[0]
    ham_subset = df[df['Label'] == 'ham'].sample(
        num_spam, random_state=451
    )
    balanced_df = pd.concat([
        ham_subset, df[df['Label'] == 'spam']
    ])
    return balanced_df

balanced_df = create_balanced_dataset(df)
balanced_df['Label'].value_counts()

Label
ham     747
spam    747
Name: count, dtype: int64

In [10]:
balanced_df['Label'] = balanced_df['Label'].map({'ham': 0, 'spam': 1})

In [13]:
def random_split(df, train_frac, val_frac):
    df = df.sample(
        frac=1, random_state=451
    ).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    val_end = train_end + int(len(df) * val_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:val_end]
    test_df = df[val_end:]

    return train_df, validation_df, test_df

train_df, val_df, test_df = random_split(
    balanced_df, 0.7, 0.1)

In [14]:
train_df.to_csv('../text/sms_spam_collection/train.csv', index=None)
val_df.to_csv('../text/sms_spam_collection/val.csv', index=None)
test_df.to_csv('../text/sms_spam_collection/test.csv', index=None)